# ClickHouse adaptive MRL scale check

Runs the real `service/app/search/engine.search()` path end-to-end (not just the
backend adapter directly, unlike notebook 09) with `adaptive_mrl.enabled=true`, against a
synthetic corpus written to a dedicated `dataset_id` on the live ClickHouse container.

**Scale honesty**: this notebook writes a few thousand rows as a genuine, executed
correctness-at-increasing-scale check, not the full 100K/1M-row backend benchmark the
plan's Sec.20 calls for -- that lives in `artifacts/advanced_retrieval/backends/` and is
produced by a separate script, because building/indexing a 100K-row table alone takes
tens of seconds per the repo's own prior measurement
(`docs/agents/TASKS.md` Faz 2: insert 35.6s + index build 38.7s at 100K/512d) and 1M+ is a
budget decision, not something to bury inside a notebook cell. Everything below is a real
`vector_math_latency`/`plumbing_smoke` measurement on this machine; nothing is estimated.

In [1]:
import os
import sys
import time
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SERVICE_ROOT = REPO_ROOT / "service"
if str(SERVICE_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVICE_ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8143")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "")
os.environ.setdefault("CLICKHOUSE_DB", "uav_search")
os.environ.setdefault("POSTGRES_HOST", "localhost")
os.environ.setdefault("POSTGRES_PORT", "5442")
os.environ.setdefault("POSTGRES_USER", "uav")
os.environ.setdefault("POSTGRES_PASSWORD", "uav_local_only")
os.environ.setdefault("POSTGRES_DB", "uav_search")
os.environ["ENABLED_DIMENSIONS"] = "512,256"

from app.db import clickhouse, postgres

reachable = clickhouse.health() and postgres.health()
print("clickhouse+postgres reachable:", reachable)

clickhouse+postgres reachable: True


In [2]:
import numpy as np

DATASET_ID = "advret_notebook10_scale"
N_ROWS = 2000

def unit_vector(seed: int, dimension: int) -> list[float]:
    rng = np.random.default_rng(seed)
    v = rng.standard_normal(dimension).astype(np.float32)
    return (v / np.linalg.norm(v)).tolist()

setup_summary = {"status": "NOT_RUN"}
if reachable:
    started = time.perf_counter()
    for dimension in (512, 256):
        rows = [
            {
                "segment_id": f"nb10_{i:05d}", "dataset_id": DATASET_ID, "video_id": "nb10_video",
                "t_start": float(i), "t_end": float(i + 1), "altitude_m": 10.0, "velocity_mps": 1.0,
                "gimbal_pitch": 0.0, "person_count": i % 5, "vehicle_count": i % 3, "is_night": 0,
                "embedding": unit_vector(i, dimension),
            }
            for i in range(N_ROWS)
        ]
        clickhouse.replace_vectors(DATASET_ID, dimension, rows)
    ingest_s = time.perf_counter() - started

    # engine.search()'s legacy_candidate_ids path (no active run registered here)
    # resolves candidate_ids via Postgres segments/videos, independent of the ClickHouse
    # vector rows written above -- both must exist for engine.search() to see any data.
    with postgres.connection() as conn, conn.cursor() as cur:
        cur.execute(
            "INSERT INTO datasets(dataset_id, has_telemetry, has_captions, vector_provenance) "
            "VALUES (%s, false, false, 'synthetic') ON CONFLICT (dataset_id) DO NOTHING",
            (DATASET_ID,),
        )
        cur.execute(
            "INSERT INTO videos(dataset_id, video_id) VALUES (%s, %s) ON CONFLICT DO NOTHING",
            (DATASET_ID, "nb10_video"),
        )
        for i in range(N_ROWS):
            cur.execute(
                "INSERT INTO segments(segment_id, dataset_id, video_id, t_start, t_end) "
                "VALUES (%s, %s, %s, %s, %s) ON CONFLICT (segment_id) DO UPDATE SET t_start=EXCLUDED.t_start",
                (f"nb10_{i:05d}", DATASET_ID, "nb10_video", float(i), float(i + 1)),
            )
        conn.commit()
    setup_summary = {"status": "plumbing_smoke", "rows_per_dimension": N_ROWS, "dimensions": [512, 256], "ingest_wall_time_s": round(ingest_s, 3)}
print(setup_summary)

{'status': 'plumbing_smoke', 'rows_per_dimension': 2000, 'dimensions': [512, 256], 'ingest_wall_time_s': 0.764}


In [3]:
# vector_math_latency: real engine.search() calls (the full production dispatch path,
# not the backend adapter directly) with adaptive_mrl enabled at increasing candidate_k.
from types import SimpleNamespace
from app.search import engine

def make_request(top_n: int, top_k: int = 10):
    return SimpleNamespace(
        query="synthetic probe", dataset_id=DATASET_ID, backend="clickhouse", strategy="exact", dimension=512,
        adaptive_mrl=SimpleNamespace(enabled=True, base_dim=256, top_n=top_n),
        metadata_filters={}, telemetry_filters={}, pattern="A", top_k=top_k, repeats=3,
        filter_execution_mode="pushdown", diagnose=True, explain=False,
    )

results = []
if reachable:
    for top_n in (50, 200, 500):
        response = engine.search(make_request(top_n))
        diag = response["diagnostics"]
        results.append({
            "top_n": top_n,
            "p50_ms": response["timings_stats"]["p50"],
            "p95_ms": response["timings_stats"]["p95"],
            "stage1_returned_candidate_count": diag["stage1_returned_candidate_count"],
            "final_returned_count": diag["final_returned_count"],
            "underfilled": diag["underfilled"],
        })
    for row in results:
        print(row)
else:
    print("NOT_RUN")

{'top_n': 50, 'p50_ms': 382.094, 'p95_ms': 395.278, 'stage1_returned_candidate_count': 50, 'final_returned_count': 10, 'underfilled': False}
{'top_n': 200, 'p50_ms': 384.397, 'p95_ms': 388.547, 'stage1_returned_candidate_count': 200, 'final_returned_count': 10, 'underfilled': False}
{'top_n': 500, 'p50_ms': 382.678, 'p95_ms': 394.531, 'stage1_returned_candidate_count': 500, 'final_returned_count': 10, 'underfilled': False}


In [4]:
# cleanup -- must mirror every table the setup cell wrote to (segments/videos/datasets in
# Postgres, seg_ch_{512,256} in ClickHouse), not just the top-level datasets row.
if reachable:
    for dimension in (512, 256):
        clickhouse.replace_vectors(DATASET_ID, dimension, [])
    with postgres.connection() as conn, conn.cursor() as cur:
        cur.execute("DELETE FROM segments WHERE dataset_id=%s", (DATASET_ID,))
        cur.execute("DELETE FROM videos WHERE dataset_id=%s", (DATASET_ID,))
        cur.execute("DELETE FROM datasets WHERE dataset_id=%s", (DATASET_ID,))
        conn.commit()
        cur.execute("SELECT count(*) FROM segments WHERE dataset_id=%s", (DATASET_ID,))
        pg_remaining = cur.fetchone()[0]
    ch_remaining = clickhouse.table_count(DATASET_ID, 512) + clickhouse.table_count(DATASET_ID, 256)
    print("cleanup complete, remaining ClickHouse rows =", ch_remaining, "| remaining Postgres segments =", pg_remaining)
    assert ch_remaining == 0
    assert pg_remaining == 0

cleanup complete, remaining ClickHouse rows = 0 | remaining Postgres segments = 0
